In [2]:
import yaml
import pandas as pd
import digitalhub as dh
from digitalhub import get_model

from data_preparation import *
from modelling import *

# Project creation

In [ ]:
project = dh.get_or_create_project("fair-hiring-tabular")
with open("../../metadata/aipc_dh.yaml", "r") as metadata_spec:
    aipc_dh_spec = yaml.safe_load(metadata_spec)

{'domain': 'PA', 'data_modality': 'tabular', 'ai_task': 'classification', 'ai_product_name': 'Hiring classifier', 'ai_product_description': 'This is a sklearn model which aims to classify whether or not an individual should be hired.', 'framework': 'dh', 'operations': [{'type': 'data_profiling', 'stage': 'data_preparation', 'id': 'data_profiling', 'name': 'Generate EDA profiling report', 'requirement_dimension': 'baseline', 'implementation': {'framework': 'dh', 'spec': {'path': 'data_preparation.py', 'method_name': 'data_drift_detection', 'requirements': 'requirements.txt', 'inputs': [{'data': 'hiring_data_original'}], 'outputs': [{'report': 'data_profiling_report'}]}}}, {'type': 'data_validation', 'stage': 'data_preparation', 'id': 'data_drift_detection', 'name': 'Evaluate whether the target production data are drifted from training data', 'requirement_dimension': 'baseline', 'implementation': {'framework': 'dh', 'spec': {'path': 'data_preparation.py', 'method_name': 'data_drift_statu

# Data preparation stage

## Load dataset

In [3]:
config = {"data_name": "recruitmentdataset.csv", "url": "https://raw.githubusercontent.com/AlbanaCelepija/enhanced_mlops/refs/heads/main/framework/library/use_cases/tabular/src/local_platform/artifacts/data/recruitmentdataset-2022.csv"}
data_preparation_fn = project.new_function(name="load_data",
                                   kind="python",
                                   python_version="PYTHON3_10",
                                   code_src="mini_data_preparation.py",
                                   handler="load_data",
                                   requirements=["scikit-learn"],
                                   labels=["data_preprocessing", "baseline"],)
data_preparation_fn = data_preparation_fn.run("job", parameters=config, wait=True)

2026-03-15 12:37:49,933 - INFO - Waiting for run a400402872cc402e92de8b7c4722c723 to finish...
INFO:digitalhub-core:Waiting for run a400402872cc402e92de8b7c4722c723 to finish...
2026-03-15 12:37:54,952 - INFO - Waiting for run a400402872cc402e92de8b7c4722c723 to finish...
INFO:digitalhub-core:Waiting for run a400402872cc402e92de8b7c4722c723 to finish...
2026-03-15 12:37:59,972 - INFO - Waiting for run a400402872cc402e92de8b7c4722c723 to finish...
INFO:digitalhub-core:Waiting for run a400402872cc402e92de8b7c4722c723 to finish...
2026-03-15 12:38:04,994 - INFO - Waiting for run a400402872cc402e92de8b7c4722c723 to finish...
INFO:digitalhub-core:Waiting for run a400402872cc402e92de8b7c4722c723 to finish...
2026-03-15 12:38:10,015 - INFO - Waiting for run a400402872cc402e92de8b7c4722c723 to finish...
INFO:digitalhub-core:Waiting for run a400402872cc402e92de8b7c4722c723 to finish...
2026-03-15 12:38:15,044 - INFO - Waiting for run a400402872cc402e92de8b7c4722c723 to finish...
INFO:digitalhub

In [4]:
raw_training_di = project.get_dataitem(config["data_name"])
raw_training_df = raw_training_di.as_df()
raw_training_df

,Id,gender,age,nationality,sport,ind-university_grade,ind-debateclub,ind-programming_exp,ind-international_exp,ind-entrepeneur_exp,ind-languages,ind-exact_study,ind-degree,company,decision
0,x8011e,female,24,German,Swimming,70,False,False,False,False,1,True,phd,A,True
1,x6077a,male,26,German,Golf,67,False,True,False,False,2,True,bachelor,A,False
2,x6006e,female,23,Dutch,Running,67,False,True,True,False,0,True,master,A,False
3,x2173b,male,24,Dutch,Cricket,70,False,True,False,False,1,True,master,A,True
4,x6241a,female,26,German,Golf,59,False,False,False,False,1,False,master,A,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3995,x7640e,female,28,Dutch,Running,63,False,False,False,False,0,False,master,D,False
3996,x3310f,female,27,Dutch,Tennis,62,False,False,False,True,2,True,bachelor,D,False
3997,x1202g,male,24,Belgian,Rugby,60,True,False,False,True,2,False,bachelor,D,False
3998,x1263d,female,22,Dutch,Tennis,66,False,True,False,False,1,True,bachelor,D,False


## Split train-validate-test set

In [19]:
config = {"test_size": 0.2, "valid_size": 0.25, "random_state": 42} # 0.25 x 0.8 = 0.2

split_fn = project.new_function(
    name="split-train-valid-test",
    kind="python",
    python_version="PYTHON3_10",
    code_src="mini_data_preparation.py",
    handler="split_train_valid_test_data",
    requirements=["scikit-learn", "numpy<2"],
    labels=["data_preprocessing", "baseline"]
)
split_run = split_fn.run(action="job", inputs={"data": raw_training_di.key}, parameters=config, wait=True)


2026-03-10 14:54:40,092 - INFO - Waiting for run a67261169d0042268a9b47f60ff0bdd7 to finish...
INFO:digitalhub-core:Waiting for run a67261169d0042268a9b47f60ff0bdd7 to finish...
2026-03-10 14:54:45,105 - INFO - Waiting for run a67261169d0042268a9b47f60ff0bdd7 to finish...
INFO:digitalhub-core:Waiting for run a67261169d0042268a9b47f60ff0bdd7 to finish...
2026-03-10 14:54:50,141 - INFO - Waiting for run a67261169d0042268a9b47f60ff0bdd7 to finish...
INFO:digitalhub-core:Waiting for run a67261169d0042268a9b47f60ff0bdd7 to finish...
2026-03-10 14:54:55,160 - INFO - Waiting for run a67261169d0042268a9b47f60ff0bdd7 to finish...
INFO:digitalhub-core:Waiting for run a67261169d0042268a9b47f60ff0bdd7 to finish...
2026-03-10 14:55:00,175 - INFO - Waiting for run a67261169d0042268a9b47f60ff0bdd7 to finish...
INFO:digitalhub-core:Waiting for run a67261169d0042268a9b47f60ff0bdd7 to finish...
2026-03-10 14:55:05,196 - INFO - Waiting for run a67261169d0042268a9b47f60ff0bdd7 to finish...
INFO:digitalhub

## Preprocess Training data

In [9]:
training_di = project.get_dataitem('training_set_X')
valid_di = project.get_dataitem('validation_set_X')
test_di = project.get_dataitem('test_set_X')
boolean_features = "ind-debateclub,ind-programming_exp,ind-international_exp,ind-entrepeneur_exp,ind-exact_study,decision"
categorical_features = "sport,ind-degree,company"
config_train = {"di_name": "train_data", "boolean_features": boolean_features, "categorical_features": categorical_features}
config_valid = {"di_name": "valid_data", "boolean_features": boolean_features, "categorical_features": categorical_features}
config_test = {"di_name": "test_data", "boolean_features": boolean_features, "categorical_features": categorical_features}

preprocess_fn = project.new_function(
    name="preprocess_train_data",
    kind="python",
    python_version="PYTHON3_10",
    code_src="dh_data_preparation.py",
    handler="preprocess_train_data_real",
    requirements=["scikit-learn", "numpy<2"],
    labels=["data_preprocessing", "baseline"]
)
preprocess_fn.run(action="job", inputs={"training_di": training_di.key}, parameters=config_train, wait=True)
preprocess_fn.run(action="job", inputs={"training_di": valid_di.key}, parameters=config_valid, wait=True)
preprocess_fn.run(action="job", inputs={"training_di": test_di.key}, parameters=config_test, wait=True)

2026-03-16 14:18:23,847 - INFO - Waiting for run 97a95496ae294d2890e5341c1c9d77db to finish...
INFO:digitalhub-core:Waiting for run 97a95496ae294d2890e5341c1c9d77db to finish...
2026-03-16 14:18:28,858 - INFO - Waiting for run 97a95496ae294d2890e5341c1c9d77db to finish...
INFO:digitalhub-core:Waiting for run 97a95496ae294d2890e5341c1c9d77db to finish...
2026-03-16 14:18:33,877 - INFO - Waiting for run 97a95496ae294d2890e5341c1c9d77db to finish...
INFO:digitalhub-core:Waiting for run 97a95496ae294d2890e5341c1c9d77db to finish...
2026-03-16 14:18:38,888 - INFO - Waiting for run 97a95496ae294d2890e5341c1c9d77db to finish...
INFO:digitalhub-core:Waiting for run 97a95496ae294d2890e5341c1c9d77db to finish...
2026-03-16 14:18:43,907 - INFO - Waiting for run 97a95496ae294d2890e5341c1c9d77db to finish...
INFO:digitalhub-core:Waiting for run 97a95496ae294d2890e5341c1c9d77db to finish...
2026-03-16 14:18:48,926 - INFO - Waiting for run 97a95496ae294d2890e5341c1c9d77db to finish...
INFO:digitalhub

{'kind': 'python+job:run', 'metadata': {'project': 'fair-hiring-tabular', 'name': '239a528310b945969b95408e16123dda', 'created': '2026-03-16T14:19:44.235Z', 'updated': '2026-03-16T14:20:22.243Z', 'created_by': 'acelepija@fbk.eu', 'updated_by': 'acelepija@fbk.eu', 'relationships': [{'type': 'run_of', 'dest': 'store://fair-hiring-tabular/function/python/preprocess_train_data:3b330d46ee7c454781f7cd29edc54cca'}, {'type': 'consumes', 'source': 'store://fair-hiring-tabular/run/python+job:run/239a528310b945969b95408e16123dda:239a528310b945969b95408e16123dda', 'dest': 'store://fair-hiring-tabular/dataitem/table/test_set_X:606c6f88e5df4580827d98d1df5dd2d2'}]}, 'spec': {'task': 'python+job://fair-hiring-tabular/b89bb03aa831433ba0d9965edd68201e', 'local_execution': False, 'function': 'python://fair-hiring-tabular/preprocess_train_data:3b330d46ee7c454781f7cd29edc54cca', 'source': {'source': 'dh_data_preparation.py', 'handler': 'preprocess_train_data_real', 'base64': 'aW1wb3J0IG9zCmltcG9ydCBwaWNrbG

# Modelling stage

In [24]:
config={"sensitive_features": ["nationality", "gender"], "random_state":42, "id_feature": "Id", "target_feature": "decision"}
train_fn = project.new_function(
    name="train-classifier",
    kind="python",
    python_version="PYTHON3_10",
    code_src="mini_modelling.py",
    handler="train_model",
    requirements=["scikit-learn", "numpy<2"],
    labels=["model_training", "baseline"],
)
train_ds = project.get_dataitem('train_data')
train_run = train_fn.run(action="job", 
                         inputs={"data_train": train_ds.key},
                         parameters=config, 
                         wait=True
                        )

2026-03-10 15:18:17,662 - INFO - Waiting for run 1a4689020b0a48d1b6bfd6dd4d39a489 to finish...
INFO:digitalhub-core:Waiting for run 1a4689020b0a48d1b6bfd6dd4d39a489 to finish...
2026-03-10 15:18:22,678 - INFO - Waiting for run 1a4689020b0a48d1b6bfd6dd4d39a489 to finish...
INFO:digitalhub-core:Waiting for run 1a4689020b0a48d1b6bfd6dd4d39a489 to finish...
2026-03-10 15:18:27,695 - INFO - Waiting for run 1a4689020b0a48d1b6bfd6dd4d39a489 to finish...
INFO:digitalhub-core:Waiting for run 1a4689020b0a48d1b6bfd6dd4d39a489 to finish...
2026-03-10 15:18:32,712 - INFO - Waiting for run 1a4689020b0a48d1b6bfd6dd4d39a489 to finish...
INFO:digitalhub-core:Waiting for run 1a4689020b0a48d1b6bfd6dd4d39a489 to finish...
2026-03-10 15:18:37,731 - INFO - Waiting for run 1a4689020b0a48d1b6bfd6dd4d39a489 to finish...
INFO:digitalhub-core:Waiting for run 1a4689020b0a48d1b6bfd6dd4d39a489 to finish...
2026-03-10 15:18:42,747 - INFO - Waiting for run 1a4689020b0a48d1b6bfd6dd4d39a489 to finish...
INFO:digitalhub

In [ ]:
model = train_run.output("model")
#model = project.get_model("hiring_classifier")
model


{'kind': 'sklearn', 'metadata': {'project': 'fair-hiring-tabular', 'name': 'hiring_classifier', 'version': 'agile-rabbit', 'created': '2026-03-10T15:18:26.716Z', 'updated': '2026-03-10T15:18:27.084Z', 'created_by': 'acelepija@fbk.eu', 'updated_by': 'acelepija@fbk.eu', 'embedded': False, 'relationships': [{'type': 'produced_by', 'dest': 'store://fair-hiring-tabular/run/python+job:run/1a4689020b0a48d1b6bfd6dd4d39a489:1a4689020b0a48d1b6bfd6dd4d39a489'}, {'type': 'produced_by', 'dest': 'store://fair-hiring-tabular/run/python+job:run/1a4689020b0a48d1b6bfd6dd4d39a489:1a4689020b0a48d1b6bfd6dd4d39a489'}]}, 'spec': {'path': 's3://datalake/fair-hiring-tabular/model/hiring_classifier/89eaac49899a4fd2a182bb05d2ea31b6/', 'parameters': {}}, 'status': {'state': 'READY', 'files': [{'path': 'model_baseline.pkl', 'name': 'model_baseline.pkl', 'content_type': 'binary/octet-stream', 'size': 883, 'hash': 'md5:2e5949167847719b1fba14e3e6f1afae', 'last_modified': '2026-03-10T15:18:26+00:00'}], 'metrics': {}},

## Evaluate overall performance metrics

In [32]:
config={"sensitive_features": ["nationality", "gender"], "id_feature": "Id", "target_feature": "decision"}
valid_ds = project.get_dataitem('valid_data')

model_evaluation_accuracy_overall_fn = project.new_function(
    name="model-evaluation-accuracy-overall",
    kind="python",
    python_version="PYTHON3_10",
    code_src="mini_modelling.py",
    handler="model_evaluation_accuracy_overall",
    requirements=["scikit-learn", "numpy<2"],
    labels=["model_evaluation", "baseline"],
)
model_evaluation_accuracy_overall_run = model_evaluation_accuracy_overall_fn.run(
    action="job",
    inputs={"model": model.key, "data_valid": valid_ds.key}, 
    parameters=config, 
    wait=True
)

2026-03-10 15:37:37,174 - INFO - Waiting for run 04fcc9ec0ed244a4b8601d392797dd55 to finish...
INFO:digitalhub-core:Waiting for run 04fcc9ec0ed244a4b8601d392797dd55 to finish...
2026-03-10 15:37:42,186 - INFO - Waiting for run 04fcc9ec0ed244a4b8601d392797dd55 to finish...
INFO:digitalhub-core:Waiting for run 04fcc9ec0ed244a4b8601d392797dd55 to finish...
2026-03-10 15:37:47,205 - INFO - Waiting for run 04fcc9ec0ed244a4b8601d392797dd55 to finish...
INFO:digitalhub-core:Waiting for run 04fcc9ec0ed244a4b8601d392797dd55 to finish...
2026-03-10 15:37:52,222 - INFO - Waiting for run 04fcc9ec0ed244a4b8601d392797dd55 to finish...
INFO:digitalhub-core:Waiting for run 04fcc9ec0ed244a4b8601d392797dd55 to finish...
2026-03-10 15:37:57,241 - INFO - Waiting for run 04fcc9ec0ed244a4b8601d392797dd55 to finish...
INFO:digitalhub-core:Waiting for run 04fcc9ec0ed244a4b8601d392797dd55 to finish...
2026-03-10 15:38:02,254 - INFO - Waiting for run 04fcc9ec0ed244a4b8601d392797dd55 to finish...
INFO:digitalhub

## Evaluate performance metrics for each demographic group

In [34]:
config={"sensitive_features": ["nationality", "gender"], "id_feature": "Id", "target_feature": "decision"}
valid_ds = project.get_dataitem('valid_data')

model_evaluation_accuracy_demographic_groups_fn = project.new_function(
    name="model-evaluation-accuracy-demographic-groups",
    kind="python",
    python_version="PYTHON3_10",
    code_src="mini_modelling.py",
    handler="model_evaluation_accuracy_demographic_groups",
    requirements=["scikit-learn", "numpy<2"],
    labels=["model_evaluation", "baseline"],
)
model_evaluation_accuracy_demographic_groups_run = model_evaluation_accuracy_demographic_groups_fn.run(
    action="job",
    inputs={"model": model.key, "data_valid": valid_ds.key}, 
    parameters=config, 
    wait=True
)

2026-03-10 15:39:42,643 - INFO - Waiting for run f71776dd03e146ce97516a384afa766a to finish...
INFO:digitalhub-core:Waiting for run f71776dd03e146ce97516a384afa766a to finish...
2026-03-10 15:39:47,671 - INFO - Waiting for run f71776dd03e146ce97516a384afa766a to finish...
INFO:digitalhub-core:Waiting for run f71776dd03e146ce97516a384afa766a to finish...
2026-03-10 15:39:52,686 - INFO - Waiting for run f71776dd03e146ce97516a384afa766a to finish...
INFO:digitalhub-core:Waiting for run f71776dd03e146ce97516a384afa766a to finish...
2026-03-10 15:39:57,703 - INFO - Waiting for run f71776dd03e146ce97516a384afa766a to finish...
INFO:digitalhub-core:Waiting for run f71776dd03e146ce97516a384afa766a to finish...
2026-03-10 15:40:02,722 - INFO - Waiting for run f71776dd03e146ce97516a384afa766a to finish...
INFO:digitalhub-core:Waiting for run f71776dd03e146ce97516a384afa766a to finish...
2026-03-10 15:40:07,741 - INFO - Waiting for run f71776dd03e146ce97516a384afa766a to finish...
INFO:digitalhub

# Operationalisation stage

In [35]:
serve_func = project.new_function(
    name="serve-classifier",
    kind="sklearnserve",
    path=model.key,
)
serve_run = serve_func.run("serve", wait=True)
serve_run

BackendError: Backend error. Response: {"timestamp":"2026-03-10T16:17:28.973+00:00","status":500,"error":"Internal Server Error","path":"/api/v1/-/fair-hiring-tabular/functions"}.

# Inference 

In [ ]:
test_df = data_preparation_fn.output("dataset").as_df().head()
test_df = test_df.drop(columns=["decision", "Id", "gender", "nationality"])
test_df


In [ ]:
data = test_df.values.tolist()
json_payload = {"inputs": [{"name": "input-0", "shape": [5, 23], "datatype": "FP32", "data": data}]}
json_payload

In [ ]:
result = serve_run.refresh().invoke(json=json_payload).json()
print("Prediction result:")
print(result)

# Workflow orchestration

## Baseline Requirements Dimension 

In [ ]:
workflow_baseline = project.new_workflow(
    name="baseline-pipeline",
    kind="hera",
    code_src="pipeline.py",
    handler="baseline_pipeline",
)

In [ ]:
wf_build = workflow_baseline.run("build", wait=True)
wf_run = workflow_baseline.run("pipeline", wait=True)

## Fairness Requirements Dimension

In [ ]:
workflow_fairness = project.new_workflow(
    name="baseline-pipeline",
    kind="hera",
    code_src="pipeline.py",
    handler="fairness_pipeline",
)

In [ ]:
wf_build = workflow_fairness.run("build", wait=True)
wf_run = workflow_fairness.run("pipeline", wait=True)